# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive exploration of the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD). All entities such as record sets, fields, and columns are referenced using their `@id`.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # keep as the mlcroissant metadata object
print(f"{metadata.name}: {metadata.description}")
print(f"DOI: {hasattr(metadata, 'identifier') and metadata.identifier or 'N/A'} | Version: {hasattr(metadata, 'version') and metadata.version or 'N/A'}")

## 2. Data Overview
Review available record sets (by `@id`), their fields, and field `@id`s.

In [ ]:
# List all record sets and their field @ids
if hasattr(metadata, 'record_sets'):
    print(f"Total record sets: {len(metadata.record_sets)}\n")
    for record_set in metadata.record_sets:
        print(f"Record set @id: {record_set.id} (name: {getattr(record_set, 'name', '-')})")
        if hasattr(record_set, 'fields'):
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - {field.id} (name: {getattr(field, 'name', '-')})")
        print()
else:
    print("No record sets found in this dataset metadata.")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Use the record set and field `@id`s found in the previous step.

In [ ]:
# --- Parameters ---
# Find an existing record set @id from the overview above.
# In this dataset, commonly used record set ids are of the form: 'https://api.app.sen.science/frontiers/.../{uuid}'
# For demonstration, we will retrieve all record set ids programmatically:

record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        record_set_ids.append(record_set.id)

dfs = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dfs[rec_id] = df
    print(f"Loaded {len(df)} records from record set {rec_id}")

# If there are no record sets or records, show a message.
if not dfs:
    print("No data record sets loaded.")

# For illustration: show field (column) names of the first available record set.
if dfs:
    # Select the first loaded dataframe
    first_rec = record_set_ids[0]
    print(f"Columns in record set {first_rec}:")
    print(dfs[first_rec].columns.tolist())
    display(dfs[first_rec].head())

## 4. Exploratory Data Analysis (EDA)
Explore numeric fields: filter, normalize, and group data using only `@id`s.

In [ ]:
# If there's no data, skip EDA steps
if dfs:
    rec_id = record_set_ids[0]  # Analyze first available record set

    df = dfs[rec_id].copy()
    # Try to find a numeric field/column by simple heuristics (expects float/int columns)
    numeric_fields = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if not numeric_fields:
        print(f"No numeric fields found in record set {rec_id} to analyze.")
    else:
        numeric_field_id = numeric_fields[0]  # Use the first numeric field

        print(f"Using numeric field with @id '{numeric_field_id}' for filtering and normalization.")
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}.")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (
            filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field if there is one (e.g., 'description', 'accession', etc.)
        cat_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = cat_fields[0] if cat_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field to group by.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show histogram and boxplot for the numeric field in the first available record set
if dfs and numeric_fields:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    # If normalized column exists, plot its histogram
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True)
        plt.title(f"Normalized {numeric_field_id}")
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
In this notebook, we loaded a FAIR^2-certified mass spectrometry dataset, explored its record sets and fields by `@id`, performed numeric filtering and basic normalization, and visualized the distributions. You can extend these analyses further by investigating specific record sets or combining multiple fields of interest. For more advanced usage and documentation, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/).
